# Server Query with OSRM
An OSRM server is deployed locally for navigation based on OSM map in England. The following code uses this service to calculate the driving time.

In [1]:
# import relevant packages
import numpy as np
import pandas as pd

from pyproj import CRS, Transformer
import requests
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# create a transformer to convert the coordinates from EPSG:27700 to EPSG:4326
crs_trans = Transformer.from_crs(CRS.from_epsg(27700), CRS.from_epsg(4326), always_xy=True)

In [3]:
%%script true
# read the station coordinates and transform them
station_raw = pd.read_csv('SimEnv/Data/station.csv', usecols=['Easting', 'Northing'])

station_coords = np.zeros((station_raw.shape[0], 2), dtype=np.float64)  # an array for the output coordinates
for i, row in x.iterrows():
    station_coords[i] = crs_trans.transform(row.iloc[0], row.iloc[1])
# station_coords = np.concatenate([station_coords[0].T, station_coords[1].T], axis=1, dtype=np.float64)

# save the result
np.savetxt('Data/temp/station_coords.txt', station_coords, fmt='%f', delimiter=',', encoding='utf-8')


In [4]:
%%script true
# This cell takes around 10 mins.
# read the event locations and transform them to EPSG:4326
event_raw = pd.read_csv('Data/wmfs_incidents_cleaned.csv', usecols=[0, 9, 10])
# print(event_raw.head())

event_coords = event_raw.copy()  # keep the first column of IDs
for i, row in event_raw.iterrows():
    if i % 10000 == 0:
        print(f'Processing row {i} of {event_raw.shape[0]}')
    event_coords.iloc[i, [1, 2]] = crs_trans.transform(row.iloc[1], row.iloc[2])
    # if i > 10000:
    #     break


# save the result
event_coords.to_csv('Data/temp/event_coords.csv', sep=',', index=False)

In [4]:
# load the station and event data
station = np.loadtxt('Data/temp/station_coords.txt', dtype=np.float64, delimiter=',')
event = np.loadtxt('Data/temp/event_coords.csv', dtype=np.float64, delimiter=',', skiprows=1, usecols=[1, 2])

print(len(station))


40


In [5]:
# define a function to parse the coordinates to required formats (normal requests in OSRM)
def parser(a : np.ndarray, b: np.ndarray) -> str:
    return str(a[0]) + ',' + str(a[1]) + ';' + str(b[0]) + ',' + str(b[1])

print(parser(station[0], event[0]))


-1.924575,52.623102;-2.051166314,52.53843293


In [6]:
# try to query the api
# define the parameters
kv = {
    'steps': 'true'
}

full_url = 'http://127.0.0.1:5010/route/v1/driving/' + parser(station[5], event[2])
print(" Final URL:", full_url)

response = requests.get(full_url, params=kv)
# try request detailed route
# response = requests.get('http://127.0.0.1:5010/route/v1/driving/' + parser(station[0], event[0]), params=kv)

response.json()

 Final URL: http://127.0.0.1:5010/route/v1/driving/-1.459379,52.404991;-1.79445107,52.55912201


{'code': 'Ok',
 'routes': [{'geometry': 'cjz~Hf_|G}iA~Fiv@clBkiAkfAm[~~Aa|BxwG{YhgBu]h{FpW|uG}Bn_FeQrlEu`@hhBeY`{C}`@lr@ajCpoAwpC~]o{@oQew@jR}YraAgPtL`\\~oAeFzqBmb@jhAbApj@',
   'legs': [{'steps': [{'intersections': [{'out': 0,
         'entry': [True],
         'bearings': [9],
         'location': [-1.459243, 52.404983]}],
       'driving_side': 'right',
       'geometry': 'cjz~Hf_|Gy@MWE',
       'mode': 'driving',
       'duration': 10.3,
       'maneuver': {'bearing_after': 9,
        'type': 'depart',
        'modifier': 'left',
        'bearing_before': 0,
        'location': [-1.459243, 52.404983]},
       'weight': 10.3,
       'distance': 46.1,
       'name': 'Mayflower Drive'},
      {'intersections': [{'out': 2,
         'location': [-1.459137, 52.405392],
         'bearings': [105, 195, 285],
         'entry': [True, False, True],
         'in': 1}],
       'driving_side': 'right',
       'geometry': 'ulz~Hr~{GGXGd@CREX',
       'mode': 'driving',
       'duration': 10.9,


In [16]:
def table_parser(loc : np.ndarray, sources_list : list):
    out1, sou, des = '', '', ''
    for i in range(loc.shape[0]):
        out1 += str(loc[i, 0]) + ',' + str(loc[i, 1]) + ';'
        if i in sources_list:
            sou += str(i) + ';'
        else:
            des += str(i) + ';'
    out1 = out1[:-1]
    out2 = {'sources': sou[:-1], 'destinations': des[:-1]}
    return out1, out2

a, b = table_parser(station[:3], [0])
print(a)
print(b)

-1.924575,52.623102;-1.893646,52.505248;-1.712608,52.453947
{'sources': '0', 'destinations': '1;2'}


In [17]:
# this cell takes around 2 mins 26 senconds
# run all stations and part of events at a time

import numpy as np
import requests
from tqdm import tqdm  


min_times = []
sample_size = 200 # the number of events to handle

for i in tqdm(range(0, event.shape[0], sample_size)):
    # the event index of the current batch
    event_list = [j for j in range(i, min(i + sample_size, event.shape[0]))]
    
    # combine the coordinate list as the type what OSRM need（station + event_batch）
    temp = np.concatenate([station, event[event_list]], axis=0)

    # URL
    locs, kv = table_parser(temp, [j for j in range(station.shape[0])])  # station 是 source

    try:
        response = requests.get('http://127.0.0.1:5010/table/v1/driving/' + locs, params=kv)
        durations = np.array(response.json()['durations']).T  # shape: (num_events, num_stations)
        batch_min_times = np.min(durations, axis=1)
        min_times.append(batch_min_times)
    except Exception as e:
        print(f" Error in batch {i}: {e}")

min_times_array = np.concatenate(min_times)

min_times_array



100%|██████████| 2074/2074 [02:26<00:00, 14.18it/s]


array([137.4,  79. , 455.7, ..., 433.3, 181.9, 279.2])

In [ ]:
# save as npy
np.save('Data/temp/min_osrm_time.npy', min_times_array)
print(" Done. Shape:", min_times_array.shape)

In [ ]:
%%script true
# this cell takes around 15-20 minutes
# run all stations and part of events at a time

out = np.zeros((0, station.shape[0]), dtype=np.int32)

sample_size = 200  # the number of events to handle

for i in range(0, event.shape[0], sample_size):
    if i % 1000 == 0:
        print(i)
    # prepare event list
    event_list = [j for j in range(i, min(i + sample_size, event.shape[0]))]
    temp = np.concatenate([station, event[event_list]], axis=0)
    locs, kv = table_parser(temp, [j for j in range(station.shape[0])])

    # try to request driving time
    response = requests.get('http://127.0.0.1:5010/table/v1/car/' + locs, params=kv)
    out1 = response.json()['durations']
    out1 = np.array(out1).T  # this is the driving time cost from all stations to a fire event
    out = np.concatenate([out, out1])
    # if i > 1000:
    #     break

# save the result
np.savetxt('/Users/zhaoyuxin/Repos/fire_station_optimisation_ga/utils/Data/temp/drv_time_osrm_raw1.csv', out, fmt='%d', delimiter=',', encoding='utf-8')


0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000
76000
77000
78000
79000
80000
81000
82000
83000
84000
85000
86000
87000
88000
89000
90000
91000
92000
93000
94000
95000
96000
97000
98000
99000
100000
101000
102000
103000
104000
105000
106000
107000
108000
109000
110000
111000
112000
113000
114000
115000
116000
117000
118000
119000
120000
121000
122000
123000
124000
125000
126000
127000
128000
129000
130000
131000
132000
133000
134000
135000
136000
137000
138000
139000
140000
141000
142000
143000
144000
145000
146000
147000
148000
149000
150000
151000
152000
153000
154000
155000
156000
157000
158000


In [12]:
# load and polish the result table
table = np.loadtxt('Data/temp/drv_time_osrm_raw1.csv', dtype=np.int32, delimiter=',', encoding='utf-8')
# add another column of the nearest station IDs for each event site as the first column
nearest_station_ids = np.argmin(table, axis=1)
min_time = np.min(table, axis=1)

# Build a pandas Dataframe from the np.array data
output = pd.DataFrame(data=table, columns=[f'Station {i}' for i in range(station.shape[0])])

event2 = pd.read_csv('Data/temp/event_coords.csv')  # load event data from the csv file again
station2 = pd.read_csv('Data/station_locations.csv')

print(station2.index)

# Four new columns are added, i.e. the event IDs, the nearest station IDs and names, and the corresponding driving time
output.insert(0, 'Incident_Number', event2.loc[:, 'Incident_Number'])
output.insert(1, 'Nearest_Station_ID', nearest_station_ids)
output.insert(2, 'Nearest_Station_Name', None)
for i in range(output.shape[0]):
    output.iloc[i, 2] = station2.loc[nearest_station_ids[i], 'Station name']
output.insert(3, 'Min_Driving_time', min_time)
output.head()

# save the result
output.to_csv('Data/temp/drv_time_osrm.csv', index=False, encoding='utf-8')

RangeIndex(start=0, stop=40, step=1)
